In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


df = pd.read_csv("iot_data.csv")

print("Första 5 raderna:")
df.head()

print("\nKolumner:")
print(df.columns)

print("\nInfo om datan:")
print(df.info())

print("\nSaknade värden per kolumn:")
print(df.isnull().sum())

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day

X = df[["device_id", "temperature", "humidity", "hour", "day_of_week", "month", "day"]]
y = df["alert"]

print("\nFeatures (X) första 5 rader:")
print(X.head())

print("\nTarget (y) första 5 rader:")
print(y.head())

print("\nFördelning av alert:")
print(y.value_counts())

Första 5 raderna:

Kolumner:
Index(['timestamp', 'device_id', 'temperature', 'humidity', 'alert'], dtype='object')

Info om datan:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   timestamp    1030 non-null   int64 
 1   device_id    1030 non-null   object
 2   temperature  1030 non-null   int64 
 3   humidity     1030 non-null   int64 
 4   alert        1030 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 40.4+ KB
None

Saknade värden per kolumn:
timestamp      0
device_id      0
temperature    0
humidity       0
alert          0
dtype: int64

Features (X) första 5 rader:
  device_id  temperature  humidity  hour  day_of_week  month  day
0  sensor_1           24        19     0            3      1    1
1  sensor_1           24        19     0            3      1    1
2  sensor_1           24        19     0            3      1    1


In [19]:


df = pd.read_csv("iot_data.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day

feature_cols = ["device_id", "temperature", "humidity", "hour", "day_of_week", "month", "day"]
X = df[feature_cols].copy()
y = df["alert"].copy()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

numeric_features = ["temperature", "humidity", "hour", "day_of_week", "month", "day"]
categorical_features = ["device_id"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_test)
print("Accuracy:", accuracy)

df["predicted_alert_encoded"] = model.predict(X)
df["predicted_alert"] = label_encoder.inverse_transform(df["predicted_alert_encoded"])

if len(label_encoder.classes_) == 2:
    probs = model.predict_proba(X)[:, 1]
    df["prediction_probability"] = probs
else:
    prob_matrix = model.predict_proba(X)
    max_probs = prob_matrix.max(axis=1)
    df["prediction_probability"] = max_probs

df.to_csv("iot_data_powerbi.csv", index=False)

print("Filen skapad: iot_data_powerbi.csv")
print(df.head())

Accuracy: 1.0
Filen skapad: iot_data_powerbi.csv
                      timestamp device_id  temperature  humidity  alert  hour  \
0 1970-01-01 00:00:01.776782111  sensor_1           24        19      0     0   
1 1970-01-01 00:00:01.776782113  sensor_1           24        19      0     0   
2 1970-01-01 00:00:01.776782116  sensor_1           24        19      0     0   
3 1970-01-01 00:00:01.776782118  sensor_1           24        19      0     0   
4 1970-01-01 00:00:01.776782120  sensor_1           24        19      0     0   

   day_of_week  month  day  predicted_alert_encoded  predicted_alert  \
0            3      1    1                        0                0   
1            3      1    1                        0                0   
2            3      1    1                        0                0   
3            3      1    1                        0                0   
4            3      1    1                        0                0   

   prediction_probability  
0  

In [20]:
df = pd.read_csv("iot_data_powerbi.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df["date"] = df["timestamp"].dt.date
df["time"] = df["timestamp"].dt.strftime("%H:%M:%S")
df["year"] = df["timestamp"].dt.year
df["week"] = df["timestamp"].dt.isocalendar().week.astype(int)

if "alert" in df.columns and "predicted_alert" in df.columns:
    df["alert_match"] = (df["alert"].astype(str) == df["predicted_alert"].astype(str)).astype(int)
else:
    df["alert_match"] = None

if "temperature" in df.columns and "humidity" in df.columns:
    df["risk_score"] = (df["temperature"] * 0.6 + df["humidity"] * 0.4).round(2)
else:
    df["risk_score"] = None

columns_order = [
    "timestamp",
    "date",
    "time",
    "year",
    "month",
    "week",
    "day",
    "day_of_week",
    "hour",
    "device_id",
    "temperature",
    "humidity",
    "alert",
    "predicted_alert",
    "prediction_probability",
    "alert_match",
    "risk_score"
]

existing_columns = [col for col in columns_order if col in df.columns]
remaining_columns = [col for col in df.columns if col not in existing_columns]

df = df[existing_columns + remaining_columns]

df.to_csv("iot_data_powerbi_final.csv", index=False)

print("Filen skapad: iot_data_powerbi_final.csv")
print(df.head())
print(df.columns)

Filen skapad: iot_data_powerbi_final.csv
                      timestamp        date      time  year  month  week  day  \
0 1970-01-01 00:00:01.776782111  1970-01-01  00:00:01  1970      1     1    1   
1 1970-01-01 00:00:01.776782113  1970-01-01  00:00:01  1970      1     1    1   
2 1970-01-01 00:00:01.776782116  1970-01-01  00:00:01  1970      1     1    1   
3 1970-01-01 00:00:01.776782118  1970-01-01  00:00:01  1970      1     1    1   
4 1970-01-01 00:00:01.776782120  1970-01-01  00:00:01  1970      1     1    1   

   day_of_week  hour device_id  temperature  humidity  alert  predicted_alert  \
0            3     0  sensor_1           24        19      0                0   
1            3     0  sensor_1           24        19      0                0   
2            3     0  sensor_1           24        19      0                0   
3            3     0  sensor_1           24        19      0                0   
4            3     0  sensor_1           24        19      0       